# Fördjupningsuppgift: Polars i Python

**Kurs:** Avancerad Python för Data Scientist, EC Utbildning

## Inledning.

I denna uppgift valde jag att titta
I denna uppgift vill jag prata om **Polars**, som är ett bibliotek i Python för att jobba med data som är ungefär som Pandas, fast nyare. Jag har hört flera gånger att Polars ska vara snabbare än Pandas. Så ville jag själv se och testa hur den fungerar i  praktiken.

Jag använder min egen data som består av elförbrukning för fem svenska städer under 2025.
Jgg valde just det här ämnet för att jag jobbade inom elbranchen i några år. Jag jobbade blandannat för E.ON. Så kändes som ett ämne jag förstår mig på lite grann.

I den här notebooken går jag igenom:
- Hur man läser in data i Polars
- Hur man filtrerar och sorterar
- Hur man räknar ut nya kolumner
- Hur man grupperar och summerar data
- En kort jämförelse med Pandas

## Vad är Polars?

Polars är ett bibliotek för att läsa och arbeta med data i tabellform, ungefär som ett Excel-ark, men i kod. Det liknar 
Pandas som vi använt tidigare i kursen, men ska enligt det jag läst mig till vara snabbare, särskilt när man har mycket data. 
Bland annat beroende på att Polars är byggt i ett annat programmeringsspråk (Rust) och kan använda flera kärnor i datorn samtidigt, 
medan Pandas oftast bara använder en.

Jag tänkte inte gå in jättedjupt i det tekniska, utan mest testa det praktiskt och se hur det känns att använda.


In [ ]:
# Insallera polars och pandas om du inte redan har dem installerade
# %pip install polars pandas
import polars as pl
import pandas as pd

print("Polars version:", pl.__version__)
print("Pandas version:", pd.__version__)

## Läser in data

Jag har en CSV-fil som heter `elforbrukninng.csv`. Den innehåller sex kolumner:
stad, region, sektor, hushåll, industri, kommersiell, elförbrukning i kWh, pris per kWh och temperatur(Elförbrukning (kWh): \(\text{Effekt (W)} \div 1\,000 \times \text{tid (timmar)} = \text{kWh}\))

I polars läser man in en fil ungefär som i Pandas, ast man skriver `pl.read_csv` iställer för `pd.read_csv`

In [ ]:
df = pl.read_csv("data/elforbrukning.csv")
df.head()

Jag kollar snabbt hur många rader och kolumner det ää, vilka datatyper det handlar om, t.ex tal, taxt osv.

In [ ]:
print("Antal rader:", df.height)
print("Antal kolumner:", df.width)
print(df.schema)

Jag kollar också om det är saknade värden någonstans, eftersom jag vet att jag lagt in några sanader temeperaturvärden med filt i datan.

In [ ]:
df.null_count()

Där ser man att det saknas 40 temperaturvärden. Det kan man behöva tänka på om man ska räkna med den
kolumnen senare.

## Filtrera data

Om jag bara vill titta på en del av datan kan jag filtrera. Här tar jag fram alla rader för Stockholm.

In [ ]:
stockholm = df.filter(pl.col("region") == "Stockholm")
stockholm.head()

Man kan också kombinera flera villkor. Här filtrerar jag fram dagar i Stockholm där förbrukningen var
över 400 kWh.

In [ ]:
stockholm_hog = df.filter(
    (pl.col("region") == "Stockholm") & (pl.col("forbrukning_kwh") > 400)
)
print("Antal rader som matchar:", stockholm_hog.height)
stockholm_hog.head()

## Sortera data

Med `.sort()` kan jag sortera datan. Här sorterar jag fram de tio dagarna med högst förbrukning.

In [ ]:
topp10 = df.sort("forbrukning_kwh", descending=True).head(10)
topp10.select(["datum", "region", "sektor", "forbrukning_kwh"])

## Räkna ut en ny kolumn

Jag vill räkna ut vad varje rad kostade, alltså förbrukning gånger pris. Det gör jag med `.with_columns()`.

In [ ]:
df_kostnad = df.with_columns(
    (pl.col("forbrukning_kwh") * pl.col("pris_sek_per_kwh")).alias("kostnad_sek")
)
df_kostnad.select(["datum", "region", "forbrukning_kwh", "pris_sek_per_kwh", "kostnad_sek"]).head()

## Gruppera och summera

Det här tyckte jag var den mest användbara delen. Med `.group_by()` och `.agg()` kan man summera eller ta
medelvärde för olika grupper. Här räknar jag ut total förbrukning per stad.

In [ ]:
per_stad = (
    df.group_by("region")
    .agg(pl.col("forbrukning_kwh").sum().alias("total_forbrukning_kwh"))
    .sort("total_forbrukning_kwh", descending=True)
)
per_stad

Jag testade också att gruppera på både stad och sektor samtidigt, för att se vilken kombination som drar
mest el.

In [ ]:
per_stad_sektor = (
    df.group_by(["region", "sektor"])
    .agg(pl.col("forbrukning_kwh").sum().alias("total_forbrukning_kwh"))
    .sort("total_forbrukning_kwh", descending=True)
)
per_stad_sektor.head(5)

## Jämförelse med Pandas

Jag ville också se hur samma uppgift ser ut i Pandas, som vi använt mer i kursen, för att jämföra. Jag tar
samma exempel som ovan: total förbrukning per stad.

In [ ]:
# Polars
print("POLARS:")
print(
    df.group_by("region")
    .agg(pl.col("forbrukning_kwh").sum().alias("total_kwh"))
    .sort("total_kwh", descending=True)
)

In [ ]:
# Pandas
df_pd = pd.read_csv("data/elforbrukning.csv")

print("PANDAS:")
print(
    df_pd.groupby("region")["forbrukning_kwh"]
    .sum()
    .reset_index(name="total_kwh")
    .sort_values("total_kwh", ascending=False)
)

Resultatet blev samma i båda, vilket kändes bra (då hade jag i alla fall inte gjort något fel). Jag tycker
Polars-koden var lite kortare och enklare att läsa rakt av, men Pandas kändes samtidigt bekant eftersom vi kört
det mer i kursen. Skillnaden i hur man skriver koden är egentligen inte jättestor när man väl ser dem bredvid
varandra.

Jag testade även att snabbt jämföra hur lång tid det tog att läsa in och gruppera datan i respektive bibliotek,
bara för att se om jag kunde märka någon skillnad.

In [ ]:
import time

start = time.time()
df.group_by("region").agg(pl.col("forbrukning_kwh").sum())
polars_tid = time.time() - start

start = time.time()
df_pd.groupby("region")["forbrukning_kwh"].sum()
pandas_tid = time.time() - start

print(f"Polars tog {polars_tid:.4f} sekunder")
print(f"Pandas tog {pandas_tid:.4f} sekunder")

På min lilla fil är skillnaden inte jättestor eftersom datan inte är så mycket, men Polars var ändå snabbare
i mitt test. Jag har läst att skillnaden blir mycket större ju mer data man har, men det testade jag inte
eftersom min fil är ganska liten.

## Sammanfattning

Jag tycker det här gav mig en bra första inblick i Polars. Det kändes inte jättekonstigt att lära sig eftersom
jag redan kan Pandas sedan innan, så mycket av logiken (läsa in, filtrera, gruppera) var densamma, bara med lite
annorlunda skrivsätt. Det jag tar med mig mest är att Polars använder `pl.col("namn")` för att peka ut vilken
kolumn man menar, vilket gjorde koden ganska tydlig att läsa.

Jag skulle gärna testa Polars mer i framtiden, särskilt om jag någon gång jobbar med större datamängder, eftersom
det verkar vara där den stora fördelen finns.

## Källor
- Polars dokumentation: https://docs.pola.rs/
- Polars på GitHub: https://github.com/pola-rs/polars
- Elförbrukning: https://www.scb.se/statistikdatabasen/kom-igang-med-statistikdatabasen/
